# Advanced Synthetic PII Data Generation (Folder-Based)
This notebook generates a vast synthetic dataset by processing all background images in our `public/` subfolders.

### Pipeline:
1.  **Image Loading**: Iterates through each category folder (`drivers`, `insurance`, `passport`, `ssn`).
2.  **PII Overlay**: Renders fake PII into appropriate locations (based on category).
3.  **Advanced Augmentations**: Applies random **Rotation**, **Cropping**, **Grayscale**, **Contrast**, and **Blur**.
4.  **Bounding Box Transformation**: Correctly updates the PaliGemma coordinates (loc tokens) to follow the image transformations.

In [21]:
!pip install -q faker pillow numpy
from PIL import Image, ImageDraw, ImageFont, ImageEnhance, ImageFilter
from faker import Faker
import random
import os
import json
import glob
import numpy as np

fake = Faker()
os.makedirs("dataset/images", exist_ok=True)
os.makedirs("dataset/labels", exist_ok=True)

# USER CONFIG: Increase this to make the dataset 'vaster'
SAMPLES_PER_BACKGROUND = 2

def apply_advanced_aug(img, bboxes):
    W, H = img.size
    
    # 1. Random Rotation (-4 to 4 degrees)
    angle = random.uniform(-4, 4)
    img = img.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor="white")
    
    # 2. Random Cropping (Zoomed 90% to 100%)
    zoom = random.uniform(0.9, 1.0)
    new_w, new_h = int(W * zoom), int(H * zoom)
    left, top = random.randint(0, W - new_w), random.randint(0, H - new_h)
    img = img.crop((left, top, left + new_w, top + new_h)).resize((W, H))
    
    # 3. Random Color & Noise
    if random.random() > 0.5: img = ImageEnhance.Brightness(img).enhance(random.uniform(0.8, 1.2))
    if random.random() > 0.5: img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))
    if random.random() > 0.3: img = img.convert("L").convert("RGB")
    if random.random() > 0.7: img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 0.7)))
    
    # Update Boxes
    updated_labels = []
    cx, cy = W/2, H/2
    rad = np.radians(-angle)
    
    for bbox in bboxes:
        x1, y1, x2, y2, label = bbox
        def rot(x, y):
            nx = cx + (x - cx) * np.cos(rad) - (y - cy) * np.sin(rad)
            ny = cy + (x - cx) * np.sin(rad) + (y - cy) * np.cos(rad)
            return nx, ny
        corners = [rot(x1, y1), rot(x2, y1), rot(x1, y2), rot(x2, y2)]
        x1r, y1r = min(p[0] for p in corners), min(p[1] for p in corners)
        x2r, y2r = max(p[0] for p in corners), max(p[1] for p in corners)
        
        # Crop Transform
        x1c, y1c = (x1r - left)/zoom, (y1r - top)/zoom
        x2c, y2c = (x2r - left)/zoom, (y2r - top)/zoom
        
        # Normalize to 1000
        lx1, ly1 = int(max(0, min(999, x1c/W*1000))), int(max(0, min(999, y1c/H*1000)))
        lx2, ly2 = int(max(0, min(999, x2c/W*1000))), int(max(0, min(999, y2c/H*1000)))
        updated_labels.append(f"<loc{ly1}><loc{lx1}><loc{ly2}><loc{lx2}> {label}")
        
    return img, " ".join(updated_labels)

print("Engine Ready.")

Engine Ready.


## 1. Drivers Category Generator
Fetches all backgrounds from `public/drivers-mock/`

In [22]:
backgrounds = glob.glob("public/drivers-mock/*.[pj]*") + glob.glob("public/drivers-mock/*.gif")
print(f"Found {len(backgrounds)} driver backgrounds.")

for bg_path in backgrounds:
    base_name = os.path.basename(bg_path).split('.')[0]
    img_orig = Image.open(bg_path).convert("RGB")
    W, H = img_orig.size
    
    for s in range(SAMPLES_PER_BACKGROUND):
        img = img_orig.copy()
        draw = ImageDraw.Draw(img)
        
        data = [
            (fake.name().upper(), "person_name", (int(W*0.3), int(H*0.4))),
            (fake.date_of_birth().strftime("%m/%d/%Y"), "dob", (int(W*0.3), int(H*0.5))),
            (fake.bothify(text='D#######'), "national_id", (int(W*0.3), int(H*0.3)))
        ]
        
        bboxes = []
        for txt, lbl, pos in data:
            draw.text(pos, txt, fill=(0,0,0))
            b = draw.textbbox(pos, txt)
            bboxes.append((b[0], b[1], b[2], b[3], lbl))
            
        img_final, label_str = apply_advanced_aug(img, bboxes)
        img_final.save(f"dataset/images/driver_{base_name}_{s}.png")
        with open(f"dataset/labels/driver_{base_name}_{s}.txt", "w") as f:
            f.write(label_str)
            
print("Driver samples generated.")

Found 8 driver backgrounds.
Driver samples generated.


## 2. Insurance Category Generator
Fetches all backgrounds from `public/insurance-mock/`

In [23]:
backgrounds = glob.glob("public/insurance-mock/*.[pj]*")
print(f"Found {len(backgrounds)} insurance backgrounds.")

for bg_path in backgrounds:
    base_name = os.path.basename(bg_path).split('.')[0]
    img_orig = Image.open(bg_path).convert("RGB")
    W, H = img_orig.size
    
    for s in range(SAMPLES_PER_BACKGROUND):
        img = img_orig.copy()
        draw = ImageDraw.Draw(img)
        data = [
            (fake.name().upper(), "person_name", (int(W*0.1), int(H*0.6))),
            (fake.bothify(text='ABC#######'), "national_id", (int(W*0.1), int(H*0.4)))
        ]
        bboxes = []
        for txt, lbl, pos in data:
            draw.text(pos, txt, fill=(0,0,0))
            b = draw.textbbox(pos, txt)
            bboxes.append((b[0], b[1], b[2], b[3], lbl))
        
        img_final, label_str = apply_advanced_aug(img, bboxes)
        img_final.save(f"dataset/images/ins_{base_name}_{s}.png")
        with open(f"dataset/labels/ins_{base_name}_{s}.txt", "w") as f:
            f.write(label_str)

print("Insurance samples generated.")

Found 5 insurance backgrounds.
Insurance samples generated.


## 3. Passport Category Generator
Fetches all backgrounds from `public/passport-mock/`

In [24]:
backgrounds = glob.glob("public/passport-mock/*.[pj]*")
print(f"Found {len(backgrounds)} passport backgrounds.")

for bg_path in backgrounds:
    base_name = os.path.basename(bg_path).split('.')[0]
    img_orig = Image.open(bg_path).convert("RGB")
    W, H = img_orig.size
    
    for s in range(SAMPLES_PER_BACKGROUND):
        img = img_orig.copy()
        draw = ImageDraw.Draw(img)
        data = [
            (fake.last_name().upper() + " << " + fake.first_name().upper(), "person_name", (int(W*0.4), int(H*0.2))),
            (fake.date_of_birth().strftime("%d %b %Y").upper(), "dob", (int(W*0.4), int(H*0.5))),
            (fake.bothify(text='#########'), "national_id", (int(W*0.7), int(H*0.1)))
        ]
        bboxes = []
        for txt, lbl, pos in data:
            draw.text(pos, txt, fill=(0,0,0))
            b = draw.textbbox(pos, txt)
            bboxes.append((b[0], b[1], b[2], b[3], lbl))
        
        img_final, label_str = apply_advanced_aug(img, bboxes)
        img_final.save(f"dataset/images/pass_{base_name}_{s}.png")
        with open(f"dataset/labels/pass_{base_name}_{s}.txt", "w") as f:
            f.write(label_str)

print("Passport samples generated.")

Found 5 passport backgrounds.
Passport samples generated.


## 4. SSN Category Generator
Fetches all backgrounds from `public/ssn-mock/`

In [25]:
backgrounds = glob.glob("public/ssn-mock/*.[pj]*") + glob.glob("public/ssn-mock/*.gif")
print(f"Found {len(backgrounds)} ssn backgrounds.")

for bg_path in backgrounds:
    base_name = os.path.basename(bg_path).split('.')[0]
    img_orig = Image.open(bg_path).convert("RGB")
    W, H = img_orig.size
    
    for s in range(SAMPLES_PER_BACKGROUND):
        img = img_orig.copy()
        draw = ImageDraw.Draw(img)
        data = [
            (fake.name().upper(), "person_name", (int(W*0.1), int(H*0.5))),
            (fake.ssn(), "ssn", (int(W*0.3), int(H*0.7)))
        ]
        bboxes = []
        for txt, lbl, pos in data:
            draw.text(pos, txt, fill=(0,0,100))
            b = draw.textbbox(pos, txt)
            bboxes.append((b[0], b[1], b[2], b[3], lbl))
        
        img_final, label_str = apply_advanced_aug(img, bboxes)
        img_final.save(f"dataset/images/ssn_{base_name}_{s}.png")
        with open(f"dataset/labels/ssn_{base_name}_{s}.txt", "w") as f:
            f.write(label_str)

print("SSN samples generated.")

Found 5 ssn backgrounds.
SSN samples generated.
